In [20]:
## Load the data
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [21]:
# Generating ground truth
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
#!wget {PREFIX}/01-agentic-rag/code/rag_helper.py
#!wget {PREFIX}/04-evaluation/code/evaluation_utils.py

In [22]:
## Module Instructions:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [23]:
## For each page, build a JSON user prompt from its filename and content, 
# then call llm_structured with the Questions model. 
# Turn each returned question into a record labeled with the page's filename. 
# The call also returns the token usage, the same as in the lessons.

## Q1: Generating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:

- `01-agentic-rag/lessons/01-intro.md`
- `01-agentic-rag/lessons/02-environment.md`
- `01-agentic-rag/lessons/03-rag.md`
Each call returns the token usage, which most LLM APIs report on the response object (e.g. `response.usage.input_tokens` / `prompt_tokens`).

What's the average number of input tokens across these 3 calls?

In [24]:
document_subset = documents[0:3]

In [25]:
first_document = documents[0]

In [27]:
## Structure output with pydantic
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]


In [28]:
## Load OpenAI items:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [29]:
## shape the document as json
import json

user_prompt = json.dumps(first_document)

In [31]:
## create the messages:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [32]:
## call the model:

response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [33]:
result = response.output_parsed

print(result)

questions=['What is retrieval-augmented generation actually doing to help an LLM answer questions more reliably?', 'Why do LLMs sometimes give wrong answers or miss information from my own files and databases?', 'What are you building in this module, and what kind of example app will it be based on?', 'What do you cover in the first part of the module before making the system agentic?', 'How is the second part of the module different from the first one in terms of how search is done?']


In [34]:
from evaluation_utils import llm_structured

In [35]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

In [37]:
## import price helper
from evaluation_utils import calc_price

In [38]:
cost = calc_price(usage)

cost

{'input_cost': 0.0007650000000000001,
 'output_cost': 0.0005085,
 'total_cost': 0.0012735}